# Translation with `indic-translate`

Translate between English and 22 Indian languages, with a choice of output script.

| | |
|---|---|
| Endpoint | `POST /translate` (JSON) |
| Model | `indic-translate` (implicit; the endpoint is model-specific) |
| Input | `text`, `target_language`, optional `script`, optional `max_output_tokens` |
| Output | `translation`, `target_language`, `script`, `truncated`, `usage` |
| Billing | output tokens only; input is free |

Source language is detected automatically. Get a key at [console.bodhan.ai](https://console.bodhan.ai) and export it as `BODHAN_API_KEY`.

In [ ]:
%pip install -q requests==2.32.3

In [ ]:
import os, json, requests

BASE_URL = os.environ.get("BODHAN_BASE_URL", "https://api.bodhan.ai")
API_KEY = os.environ["BODHAN_API_KEY"]  # export BODHAN_API_KEY=... before starting Jupyter
HEADERS = {"Authorization": f"Bearer {API_KEY}"}


def raise_for_bodhan(resp):
    """Bodhan errors are JSON: {"error": {"message", "code", "request_id"}}. Surface them readably."""
    if resp.ok:
        return resp
    try:
        err = resp.json()["error"]
        raise RuntimeError(f"{resp.status_code} {err.get('code')}: {err.get('message')} (request_id={err.get('request_id')})")
    except (ValueError, KeyError):
        resp.raise_for_status()

## 1. A first translation

In [ ]:
resp = requests.post(
    f"{BASE_URL}/translate",
    headers={**HEADERS, "Content-Type": "application/json"},
    json={"text": "The meeting has been postponed to Monday.", "target_language": "kn"},
    timeout=60,
)
print(json.dumps(raise_for_bodhan(resp).json(), ensure_ascii=False, indent=2))

## 2. Helper

In [ ]:
def translate(text: str, target_language: str, script: str | None = None, max_output_tokens: int | None = None) -> dict:
    body = {"text": text, "target_language": target_language}
    if script:
        body["script"] = script
    if max_output_tokens:
        body["max_output_tokens"] = max_output_tokens
    resp = requests.post(f"{BASE_URL}/translate", headers={**HEADERS, "Content-Type": "application/json"}, json=body, timeout=60)
    return raise_for_bodhan(resp).json()


translate("Please submit your homework by Friday.", "hi")["translation"]

## 3. Output script: native, roman, or code-mixed

`script` controls how the target text is written:

- `native` (default) — the language's own script: देवनागरी, தமிழ், ...
- `roman` — Latin letters, the way people type on WhatsApp: *"kripya shukravaar tak..."*
- `codemix` — everyday Hinglish/Tanglish style, mixing English words where speakers naturally would

In [ ]:
for script in ("native", "roman", "codemix"):
    out = translate("Your order will be delivered tomorrow between 10 am and 12 pm.", "hi", script=script)
    print(f"{script:8} {out['translation']}")

## 4. Fan out to many languages

The same text into several languages. Requests are independent, so a thread pool is the easy speed-up; keep it modest to stay under your key's parallel-request limit.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

TARGETS = ["hi", "bn", "ta", "te", "mr", "gu", "kn", "ml", "pa", "or"]
notice = "School will remain closed tomorrow due to heavy rain."

with ThreadPoolExecutor(max_workers=4) as pool:
    results = list(pool.map(lambda lang: (lang, translate(notice, lang)["translation"]), TARGETS))

for lang, text in results:
    print(f"{lang:4} {text}")

## 5. Language codes

| Code | Language | Code | Language |
|---|---|---|---|
| `as` | Assamese | `mni` / `mni-Beng` | Manipuri (Meitei Mayek / Bengali script) |
| `bn` | Bengali | `mr` | Marathi |
| `brx` | Bodo | `ne` | Nepali |
| `doi` | Dogri | `or` | Odia |
| `en` | English | `pa` | Punjabi |
| `gu` | Gujarati | `sa` | Sanskrit |
| `hi` | Hindi | `sat` | Santali |
| `kn` | Kannada | `sd` / `sd-Arab` | Sindhi (Devanagari / Perso-Arabic) |
| `kok` | Konkani | `ta` | Tamil |
| `ks` / `ks-Deva` | Kashmiri (Perso-Arabic / Devanagari) | `te` | Telugu |
| `mai` | Maithili | `ur` | Urdu |
| `ml` | Malayalam | | |

## 6. Long inputs and `truncated`

The response carries `truncated: true` when the output hit `max_output_tokens`. For long documents, split on paragraphs, translate each, and rejoin; you keep sentence boundaries intact and can parallelise.

In [ ]:
def translate_document(text: str, target_language: str, **kw) -> str:
    paragraphs = [p for p in text.split("\n\n") if p.strip()]
    return "\n\n".join(translate(p, target_language, **kw)["translation"] for p in paragraphs)


doc = """Photosynthesis is the process by which plants make food.

It takes place in the leaves, using sunlight, water and carbon dioxide."""
print(translate_document(doc, "ta"))

**Next:** turn the translation into audio with [`text-to-speech`](../text-to-speech/text_to_speech.ipynb), or write roman-script text back to native script with [`transliterate`](../transliterate/transliterate.ipynb).